# India Climate Data Science

Reproducible end-to-end analysis of India's climate system, 1950–2022.

This notebook combines global temperature anomalies, India temperature, monsoon rainfall, ENSO/ONI indicators, and IMD gridded temperature data. It includes trend testing, correlation analysis, PCA, clustering, regression, classification, ARIMA forecasting, and IMD diurnal-temperature-range analysis.

## GitHub usage

1. Clone or download the repository.
2. Place the required datasets in the repository's `data/` folders using the structure documented in `README.md`.
3. Install dependencies from `requirements.txt`.
4. Open this notebook and run **Restart Kernel and Run All**.

The notebook uses paths relative to the repository root, so it can run on GitHub Codespaces, JupyterLab, and local environments without changing user-specific paths.


In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import importlib
from scipy.stats import linregress, pearsonr, spearmanr, shapiro, jarque_bera

from sklearn.model_selection import TimeSeriesSplit
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix,
)
from sklearn.inspection import permutation_importance

try:
    from xgboost import XGBRegressor
    HAS_XGBOOST = True
except Exception:
    HAS_XGBOOST = False

try:
    import shap
    HAS_SHAP = True
except Exception:
    HAS_SHAP = False

try:
    import pymannkendall as mk
    HAS_MK = True
except Exception:
    HAS_MK = False

try:
    import xarray as xr
    HAS_XARRAY = True
except Exception:
    HAS_XARRAY = False

try:
    from statsmodels.tsa.arima.model import ARIMA
    from statsmodels.tsa.statespace.sarimax import SARIMAX
    from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
    from statsmodels.tsa.stattools import adfuller
    from statsmodels.stats.diagnostic import acorr_ljungbox
    HAS_STATSMODELS = True
except Exception:
    HAS_STATSMODELS = False

try:
    import openpyxl
    HAS_OPENPYXL = True
except Exception:
    HAS_OPENPYXL = False

sns.set_theme(style='whitegrid', context='talk')
plt.rcParams['figure.figsize'] = (12, 7)
plt.rcParams['figure.dpi'] = 140
plt.rcParams['axes.titleweight'] = 'bold'
pd.set_option('display.max_columns', 300)
pd.set_option('display.width', 240)


In [ ]:
PROJECT_DIR = Path.cwd() / 'climate_phd_project'
if not PROJECT_DIR.exists():
    PROJECT_DIR = Path.home() / 'climate_phd_project'

DATA_DIR = PROJECT_DIR / 'data'
RAW_DIR = DATA_DIR / 'raw'
PROCESSED_DIR = DATA_DIR / 'processed'
CUSTOM_DIR = PROCESSED_DIR / 'climate_project_outputs_custom'
IMD_DIR = PROCESSED_DIR / 'imd_temperature_csv'
EXTERNAL_DIR = PROJECT_DIR / 'external'

OUTPUT_DIR = PROCESSED_DIR / 'climate_project_outputs_final_thesis_complete'
FIGURE_DIR = OUTPUT_DIR / 'figures'
EXPORT_DIR = OUTPUT_DIR / 'exports'
TABLE_DIR = OUTPUT_DIR / 'tables'

for d in [OUTPUT_DIR, FIGURE_DIR, EXPORT_DIR, TABLE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

ANALYSIS_START_YEAR = 1950
ANALYSIS_END_YEAR = 2022
INDIA_BASE_START, INDIA_BASE_END = 1951, 1980
RAINFALL_BASE_START, RAINFALL_BASE_END = 1951, 1980
GLOBAL_ABSOLUTE_BASELINE = 14.0
FORECAST_HORIZON = 10
RANDOM_STATE = 42
N_BOOTSTRAP = 2000

CONFIG = {
    'project_dir': str(PROJECT_DIR),
    'data_dir': str(DATA_DIR),
    'processed_dir': str(PROCESSED_DIR),
    'custom_dir': str(CUSTOM_DIR),
    'imd_dir': str(IMD_DIR),
    'external_dir': str(EXTERNAL_DIR),
    'output_dir': str(OUTPUT_DIR),
    'analysis_period': [ANALYSIS_START_YEAR, ANALYSIS_END_YEAR],
    'forecast_horizon': FORECAST_HORIZON,
    'has_xgboost': HAS_XGBOOST,
    'has_shap': HAS_SHAP,
    'has_mk': HAS_MK,
    'has_xarray': HAS_XARRAY,
    'has_statsmodels': HAS_STATSMODELS,
    'has_openpyxl': HAS_OPENPYXL,
}
(OUTPUT_DIR / 'configuration.json').write_text(json.dumps(CONFIG, indent=2))
CONFIG


In [ ]:
package_names = ['xgboost', 'shap', 'pymannkendall', 'xarray', 'statsmodels', 'openpyxl']
package_rows = []
for pkg in package_names:
    available = importlib.util.find_spec(pkg) is not None
    package_rows.append({'Package': pkg, 'Installed': available})
package_status_df = pd.DataFrame(package_rows)
package_status_df


In [ ]:
def read_csv_flex(path):
    df = pd.read_csv(path)
    df.columns = [str(c).strip() for c in df.columns]
    return df

def normalize_year_col(df):
    candidates = [c for c in df.columns if str(c).strip().lower() in ['year', 'yr']]
    if not candidates:
        for c in df.columns:
            if 'year' in str(c).strip().lower():
                candidates.append(c)
                break
    if not candidates:
        raise ValueError(f'No year column found. Columns: {list(df.columns)}')
    year_col = candidates[0]
    out = df.copy()
    out[year_col] = pd.to_numeric(out[year_col], errors='coerce')
    out = out.dropna(subset=[year_col]).copy()
    out[year_col] = out[year_col].astype(int)
    if year_col != 'Year':
        out = out.rename(columns={year_col: 'Year'})
    return out

def clip_years(df, start=ANALYSIS_START_YEAR, end=ANALYSIS_END_YEAR):
    out = normalize_year_col(df.copy())
    out = out[out['Year'].between(start, end)].copy()
    out = out.sort_values('Year').drop_duplicates('Year').reset_index(drop=True)
    return out

def first_existing_col(df, options):
    mapping = {str(c).strip().lower(): c for c in df.columns}
    for opt in options:
        if opt.lower() in mapping:
            return mapping[opt.lower()]
    compact = {str(c).strip().lower().replace(' ', '').replace('_', ''): c for c in df.columns}
    for opt in options:
        k = opt.lower().replace(' ', '').replace('_', '')
        if k in compact:
            return compact[k]
    return None

def ensure_numeric(df, cols):
    out = df.copy()
    for c in cols:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors='coerce')
    return out

def add_anomaly(df, value_col, baseline_start, baseline_end, new_col):
    out = df.copy()
    baseline = out.loc[out['Year'].between(baseline_start, baseline_end), value_col].mean()
    out[new_col] = out[value_col] - baseline
    return out, float(baseline) if pd.notna(baseline) else np.nan

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def safe_sheet_name(name):
    return str(name)[:31]

def save_fig(name):
    p = FIGURE_DIR / name
    plt.tight_layout()
    plt.savefig(p, dpi=260, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f'Saved figure to {p}')

def find_first_existing(paths):
    for p in paths:
        if p is not None and Path(p).exists():
            return Path(p)
    return None

def find_oni_file():
    candidates = [
        PROJECT_DIR / 'oni.csv',
        EXTERNAL_DIR / 'oni.csv',
        DATA_DIR / 'external' / 'oni.csv',
        CUSTOM_DIR / 'oni.csv',
    ]
    found = find_first_existing(candidates)
    if found:
        return found
    matches = list(PROJECT_DIR.rglob('oni.csv'))
    if matches:
        return matches[0]
    raise FileNotFoundError('oni.csv not found in project folders.')


In [ ]:
resolved = {
    'global_file': find_first_existing([
        CUSTOM_DIR / 'global_annual_clean.csv',
        PROCESSED_DIR / 'annual.csv',
        DATA_DIR / 'annual.csv',
    ]),
    'india_file': find_first_existing([
        CUSTOM_DIR / 'india_annual_clean.csv',
        PROCESSED_DIR / 'india_official.csv',
        DATA_DIR / 'india_official.csv',
    ]),
    'rainfall_file': find_first_existing([
        CUSTOM_DIR / 'rainfall_annual_clean.csv',
        PROCESSED_DIR / 'india_rainfall_annual.csv',
        DATA_DIR / 'india_rainfall_annual.csv',
    ]),
    'imd_tmax_annual': find_first_existing([
        IMD_DIR / 'india_area_weighted_tmax_annual.csv',
        IMD_DIR / 'india_area_weighted_temperature_annual.csv',
    ]),
    'imd_tmin_annual': find_first_existing([
        IMD_DIR / 'india_area_weighted_tmin_annual.csv',
    ]),
    'oni_file': find_oni_file(),
    'rainfall_nc_count': len(list(PROJECT_DIR.rglob('RF25_ind*_rfp25.nc'))),
}

file_status_df = pd.DataFrame([
    {
        'Key': k,
        'Path': str(v) if v is not None else 'NOT FOUND',
        'Exists': bool(v) if not isinstance(v, int) else v > 0,
    }
    for k, v in resolved.items()
])
file_status_df


In [ ]:
def load_global_data(path):
    df = clip_years(read_csv_flex(path))
    col = first_existing_col(df, ['GlobalAnomaly', 'AnnualMean', 'Global Temp Anomaly', 'Anomaly'])
    if col is None:
        col = [c for c in df.columns if c != 'Year'][0]
    df = ensure_numeric(df, [col])
    out = df[['Year', col]].rename(columns={col: 'GlobalAnomaly'}).copy()
    out['GlobalAbsoluteTemp'] = out['GlobalAnomaly'] + GLOBAL_ABSOLUTE_BASELINE
    return out

def load_india_data(path):
    df = clip_years(read_csv_flex(path))
    col = first_existing_col(df, ['IndiaTemp', 'AnnualMeanTemp', 'Annual Mean Temperature', 'Temperature'])
    if col is None:
        col = [c for c in df.columns if c != 'Year'][0]
    df = ensure_numeric(df, [col])
    out = df[['Year', col]].rename(columns={col: 'IndiaTemp'}).copy()
    out, baseline = add_anomaly(out, 'IndiaTemp', INDIA_BASE_START, INDIA_BASE_END, 'IndiaTempAnomaly')
    return out, baseline

def load_rainfall_data(path):
    df = clip_years(read_csv_flex(path))
    col = first_existing_col(df, ['Rainfall', 'AnnualRainfall', 'Rainfallmm', 'TotalRainfall'])
    if col is None:
        col = [c for c in df.columns if c != 'Year'][0]
    out = df[['Year', col]].rename(columns={col: 'Rainfall'}).copy()
    out = ensure_numeric(out, ['Rainfall'])
    out, baseline = add_anomaly(out, 'Rainfall', RAINFALL_BASE_START, RAINFALL_BASE_END, 'RainfallAnomaly')
    return out, baseline

def load_oni_annual(path):
    oni = read_csv_flex(path)
    date_candidates = [c for c in oni.columns if str(c).strip().lower() in ['date', 'time', 'month']]
    if not date_candidates:
        raise ValueError(f'Could not find a date-like column in ONI file. Columns: {list(oni.columns)}')
    date_col = date_candidates[0]

    numeric_candidates = []
    for c in oni.columns:
        if c == date_col:
            continue
        temp = pd.to_numeric(oni[c], errors='coerce')
        if temp.notna().sum() > 0:
            numeric_candidates.append(c)
    if not numeric_candidates:
        raise ValueError(f'Could not find a numeric ONI value column. Columns: {list(oni.columns)}')
    value_col = numeric_candidates[0]

    oni['Date'] = pd.to_datetime(oni[date_col], errors='coerce')
    oni['ONI'] = pd.to_numeric(oni[value_col], errors='coerce')
    oni.loc[oni['ONI'] <= -99, 'ONI'] = np.nan
    oni = oni.dropna(subset=['Date', 'ONI']).copy()
    oni['Year'] = oni['Date'].dt.year.astype(int)
    oni['Month'] = oni['Date'].dt.month.astype(int)

    annual = oni.groupby('Year')['ONI'].mean().rename('ONIAnnualMean')
    monsoon = oni[oni['Month'].isin([6, 7, 8, 9])].groupby('Year')['ONI'].mean().rename('ONIMonsoonMean')
    out = pd.concat([annual, monsoon], axis=1).reset_index()
    return clip_years(out), oni

global_df = load_global_data(resolved['global_file'])
india_df, india_baseline = load_india_data(resolved['india_file'])
rainfall_df, rainfall_baseline = load_rainfall_data(resolved['rainfall_file'])
oni_annual_df, oni_monthly_df = load_oni_annual(resolved['oni_file'])

print('Loaded dataframes successfully')
display(global_df.head())
display(india_df.head())
display(rainfall_df.head())
display(oni_annual_df.head())


In [ ]:
master_df = (
    global_df
    .merge(india_df, on='Year', how='outer')
    .merge(rainfall_df, on='Year', how='outer')
)
master_df = master_df.merge(oni_annual_df, on='Year', how='left')
master_df = clip_years(master_df)
master_df = master_df.loc[:, ~master_df.columns.duplicated()].copy()
master_df = master_df.sort_values('Year').drop_duplicates('Year').reset_index(drop=True)

merged_df = master_df.copy()

print('master_df created successfully')
print('Shape:', master_df.shape)
print('Year range:', int(master_df['Year'].min()), 'to', int(master_df['Year'].max()))
display(master_df.head())


In [ ]:
features_df = master_df.copy()
features_df['Decade'] = (features_df['Year'] // 10) * 10
features_df['YearIndex'] = features_df['Year'] - features_df['Year'].min()

feature_cols_for_lags = [
    'GlobalAnomaly',
    'IndiaTemp',
    'IndiaTempAnomaly',
    'Rainfall',
    'RainfallAnomaly',
    'ONIAnnualMean',
    'ONIMonsoonMean',
]
for col in feature_cols_for_lags:
    if col in features_df.columns:
        features_df[f'{col}lag1'] = features_df[col].shift(1)
        features_df[f'{col}lag2'] = features_df[col].shift(2)
        features_df[f'{col}diff1'] = features_df[col].diff(1)
        features_df[f'{col}rolling3'] = features_df[col].rolling(3).mean()
        features_df[f'{col}rolling5'] = features_df[col].rolling(5).mean()

display(features_df.head())


In [ ]:
summary_metrics = pd.DataFrame([
    {'Metric': 'Number of years', 'Value': float(master_df['Year'].nunique())},
    {'Metric': 'First year', 'Value': float(master_df['Year'].min())},
    {'Metric': 'Last year', 'Value': float(master_df['Year'].max())},
    {'Metric': 'Missing rainfall values', 'Value': float(master_df['Rainfall'].isna().sum()) if 'Rainfall' in master_df.columns else np.nan},
    {'Metric': 'Duplicate years', 'Value': float(master_df['Year'].duplicated().sum())},
    {'Metric': 'India baseline mean', 'Value': india_baseline},
    {'Metric': 'Rainfall baseline mean', 'Value': rainfall_baseline},
])
summary_metrics


In [ ]:
trend_results = []
for col in ['GlobalAnomaly', 'IndiaTemp', 'IndiaTempAnomaly', 'Rainfall', 'RainfallAnomaly', 'ONIAnnualMean', 'ONIMonsoonMean']:
    if col not in master_df.columns:
        continue
    temp = master_df[['Year', col]].dropna().copy()
    if len(temp) < 3:
        continue

    if HAS_MK:
        mk_result = mk.original_test(temp[col].to_numpy())
        trend = mk_result.trend
        p_value = float(mk_result.p)
        tau = float(mk_result.Tau)
        sen_slope = float(mk_result.slope)
    else:
        trend, p_value, tau, sen_slope = 'not tested', np.nan, np.nan, np.nan

    lr = linregress(temp['Year'], temp[col])
    trend_results.append({
        'Variable': col,
        'Trend': trend,
        'pvalue': p_value,
        'Tau': tau,
        'SenSlopePerYear': sen_slope,
        'LinearSlopePerYear': float(lr.slope),
        'Significant5pct': bool(p_value < 0.05) if pd.notna(p_value) else False,
        'N': int(len(temp)),
    })

trend_df = pd.DataFrame(trend_results)
trend_df


In [ ]:
corr_cols = ['GlobalAnomaly', 'IndiaTemp', 'IndiaTempAnomaly', 'Rainfall', 'RainfallAnomaly', 'ONIAnnualMean', 'ONIMonsoonMean']
corr_cols = [c for c in corr_cols if c in master_df.columns]
corr_matrix = master_df[corr_cols].corr(numeric_only=True)
corr_matrix


In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, ax=ax, fmt='.2f')
ax.set_title('Correlation matrix of annual climate variables')
save_fig('01_correlation_matrix.png')


In [ ]:
plot_df = master_df.copy()

fig, axes = plt.subplots(3, 1, figsize=(14, 16), sharex=True)

if 'GlobalAnomaly' in plot_df.columns:
    axes[0].plot(plot_df['Year'], plot_df['GlobalAnomaly'], marker='o', lw=2.2, label='Global anomaly')
    lr = linregress(plot_df[['Year', 'GlobalAnomaly']].dropna()['Year'], plot_df[['Year', 'GlobalAnomaly']].dropna()['GlobalAnomaly'])
    valid = plot_df[['Year', 'GlobalAnomaly']].dropna()
    axes[0].plot(valid['Year'], lr.intercept + lr.slope * valid['Year'], lw=2, ls='--', label='Linear trend')
    axes[0].set_title('Global temperature anomaly')
    axes[0].set_ylabel('Anomaly')
    axes[0].legend(frameon=True)

if 'IndiaTempAnomaly' in plot_df.columns:
    axes[1].plot(plot_df['Year'], plot_df['IndiaTempAnomaly'], marker='o', lw=2.2, color='tab:red', label='India temperature anomaly')
    valid = plot_df[['Year', 'IndiaTempAnomaly']].dropna()
    lr = linregress(valid['Year'], valid['IndiaTempAnomaly'])
    axes[1].plot(valid['Year'], lr.intercept + lr.slope * valid['Year'], lw=2, ls='--', label='Linear trend')
    axes[1].set_title('India temperature anomaly')
    axes[1].set_ylabel('Anomaly')
    axes[1].legend(frameon=True)

if 'RainfallAnomaly' in plot_df.columns:
    axes[2].bar(plot_df['Year'], plot_df['RainfallAnomaly'], color=np.where(plot_df['RainfallAnomaly'] >= 0, 'tab:blue', 'tab:orange'))
    axes[2].axhline(0, color='black', lw=1)
    axes[2].set_title('India rainfall anomaly')
    axes[2].set_ylabel('Anomaly')
    axes[2].set_xlabel('Year')

save_fig('02_main_trends.png')


In [ ]:
pca_vars = ['GlobalAnomaly', 'IndiaTempAnomaly', 'RainfallAnomaly', 'ONIAnnualMean', 'ONIMonsoonMean']
pca_vars = [c for c in pca_vars if c in features_df.columns]

pca_input = features_df[pca_vars].copy()
pca_input = pd.DataFrame(SimpleImputer(strategy='median').fit_transform(pca_input), columns=pca_vars)
pca_scaled = StandardScaler().fit_transform(pca_input)

pca = PCA(n_components=min(len(pca_vars), len(pca_vars)))
pca_scores = pca.fit_transform(pca_scaled)

explained_variance_df = pd.DataFrame({
    'PC': [f'PC{i}' for i in range(1, len(pca.explained_variance_ratio_) + 1)],
    'ExplainedVarianceRatio': pca.explained_variance_ratio_,
    'CumulativeVarianceRatio': np.cumsum(pca.explained_variance_ratio_),
})

loadings_df = pd.DataFrame(
    pca.components_.T,
    index=pca_vars,
    columns=[f'PC{i}' for i in range(1, len(pca.explained_variance_ratio_) + 1)]
).reset_index().rename(columns={'index': 'Variable'})

pca_scores_df = pd.DataFrame({
    'Year': features_df['Year'].values,
    'PC1': pca_scores[:, 0],
    'PC2': pca_scores[:, 1] if pca_scores.shape[1] > 1 else np.nan,
})

display(explained_variance_df)
display(loadings_df)
display(pca_scores_df.head())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

axes[0].bar(explained_variance_df['PC'], explained_variance_df['ExplainedVarianceRatio'], color='tab:purple')
axes[0].plot(explained_variance_df['PC'], explained_variance_df['CumulativeVarianceRatio'], marker='o', color='black')
axes[0].set_title('PCA explained variance')
axes[0].set_ylabel('Variance ratio')

scatter = axes[1].scatter(pca_scores_df['PC1'], pca_scores_df['PC2'], c=pca_scores_df['Year'], cmap='viridis', s=70)
axes[1].set_title('PCA scores')
axes[1].set_xlabel('PC1')
axes[1].set_ylabel('PC2')
plt.colorbar(scatter, ax=axes[1], label='Year')

save_fig('03_pca_summary.png')


In [ ]:
cluster_input = pca_scores_df[['PC1', 'PC2']].copy()
cluster_input = cluster_input.fillna(cluster_input.median())

kmeans = KMeans(n_clusters=3, n_init=20, random_state=RANDOM_STATE)
agg = AgglomerativeClustering(n_clusters=3)

pca_scores_df['KMeansCluster'] = kmeans.fit_predict(cluster_input)
pca_scores_df['AggCluster'] = agg.fit_predict(cluster_input)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

axes[0].scatter(pca_scores_df['PC1'], pca_scores_df['PC2'], c=pca_scores_df['KMeansCluster'], cmap='tab10', s=70)
axes[0].set_title('KMeans clusters in PCA space')
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')

axes[1].scatter(pca_scores_df['PC1'], pca_scores_df['PC2'], c=pca_scores_df['AggCluster'], cmap='tab10', s=70)
axes[1].set_title('Agglomerative clusters in PCA space')
axes[1].set_xlabel('PC1')
axes[1].set_ylabel('PC2')

save_fig('04_clustering.png')
pca_scores_df.head()


In [ ]:
model_target = 'IndiaTempAnomaly' if 'IndiaTempAnomaly' in features_df.columns else 'IndiaTemp'

target_leakage_features = {
    f'{model_target}diff1',
    f'{model_target}rolling3',
    f'{model_target}rolling5',
    'IndiaTempdiff1',
    'IndiaTemprolling3',
    'IndiaTemprolling5',
    'IndiaTempAnomalydiff1',
    'IndiaTempAnomalyrolling3',
    'IndiaTempAnomalyrolling5',
}

feature_candidates = [
    c for c in features_df.columns
    if c not in ['Year', 'Decade', model_target, 'IndiaTemp', 'IndiaTempAnomaly']
    and c not in target_leakage_features
    and pd.api.types.is_numeric_dtype(features_df[c])
]

model_df = features_df[['Year', model_target] + feature_candidates].dropna(subset=[model_target]).reset_index(drop=True)
X = pd.DataFrame(SimpleImputer(strategy='median').fit_transform(model_df[feature_candidates]), columns=feature_candidates)
y = model_df[model_target].copy()

tscv = TimeSeriesSplit(n_splits=5)
models = {
    'Ridge': Pipeline([('scaler', StandardScaler()), ('model', Ridge(alpha=1.0))]),
    'RandomForest': RandomForestRegressor(n_estimators=300, max_depth=6, random_state=RANDOM_STATE),
}
if HAS_XGBOOST:
    models['XGBoost'] = XGBRegressor(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        objective='reg:squarederror',
        random_state=RANDOM_STATE,
    )

results = []
preds = []
last_fitted_models = {}
for model_name, model in models.items():
    fold = 0
    for train_idx, test_idx in tscv.split(X):
        fold += 1
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        model.fit(X_train, y_train)
        yp = model.predict(X_test)
        results.append({
            'Model': model_name,
            'Fold': fold,
            'MAE': mean_absolute_error(y_test, yp),
            'RMSE': rmse(y_test, yp),
            'R2': r2_score(y_test, yp),
        })
        preds.append(pd.DataFrame({
            'Year': model_df.iloc[test_idx]['Year'].values,
            'Actual': y_test.values,
            'Predicted': yp,
            'Model': model_name,
            'Fold': fold,
        }))
        last_fitted_models[model_name] = model

model_results_df = pd.DataFrame(results)
predictions_df = pd.concat(preds, ignore_index=True)
model_summary_df = model_results_df.groupby('Model', as_index=False)[['MAE', 'RMSE', 'R2']].mean().sort_values('RMSE')
model_summary_df


In [ ]:
featuresdf = master_df.copy()
featuresdf["Decade"] = (featuresdf["Year"] // 10) * 10
featuresdf["YearIndex"] = featuresdf["Year"] - featuresdf["Year"].min()

basecols = [
    c for c in [
        "GlobalAnomaly",
        "IndiaTemp",
        "IndiaTempAnomaly",
        "Rainfall",
        "RainfallAnomaly",
        "ONIAnnualMean",
        "ONIMonsoonMean",
    ]
    if c in featuresdf.columns
]

for col in basecols:
    featuresdf[f"{col}lag1"] = featuresdf[col].shift(1)
    featuresdf[f"{col}lag2"] = featuresdf[col].shift(2)
    featuresdf[f"{col}diff1"] = featuresdf[col].diff(1)
    featuresdf[f"{col}rolling3"] = featuresdf[col].rolling(3).mean()
    featuresdf[f"{col}rolling5"] = featuresdf[col].rolling(5).mean()

print("featuresdf created:", featuresdf.shape)
featuresdf.head()


In [ ]:
best_model_name = model_summary_df.iloc[0]['Model']
best_pred_df = predictions_df[predictions_df['Model'] == best_model_name].sort_values('Year').copy()
resid = best_pred_df['Actual'] - best_pred_df['Predicted']

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
axes[0].plot(best_pred_df['Year'], best_pred_df['Actual'], marker='o', lw=2.3, label='Actual')
axes[0].plot(best_pred_df['Year'], best_pred_df['Predicted'], marker='s', lw=2.3, label='Predicted')
axes[0].set_title(f'Best model predictions: {best_model_name}')
axes[0].set_xlabel('Year')
axes[0].set_ylabel(model_target)
axes[0].legend(frameon=True)

axes[1].scatter(best_pred_df['Predicted'], resid, color='tab:red', s=60)
axes[1].axhline(0, color='black', lw=1)
axes[1].set_title('Residual plot')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Residual')

save_fig('07_best_model_and_residuals.png')


In [ ]:
final_best_model = last_fitted_models[best_model_name]
final_best_model.fit(X, y)
perm = permutation_importance(final_best_model, X, y, n_repeats=20, random_state=RANDOM_STATE)
importance_df = pd.DataFrame({'Feature': X.columns, 'Importance': perm.importances_mean}).sort_values('Importance', ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
importance_df.head(15).sort_values('Importance').plot(kind='barh', x='Feature', y='Importance', ax=ax, color='tab:green', legend=False)
ax.set_title('Permutation feature importance')
ax.set_xlabel('Mean importance')
save_fig('08_feature_importance.png')

importance_df.head(15)


In [ ]:
regression_diagnostic_rows = []
if len(resid) >= 3:
    regression_diagnostic_rows.append({'Diagnostic': 'Residual mean', 'Value': float(np.mean(resid))})
    regression_diagnostic_rows.append({'Diagnostic': 'Residual std', 'Value': float(np.std(resid, ddof=1))})
    try:
        sh_stat, sh_p = shapiro(resid)
        regression_diagnostic_rows.append({'Diagnostic': 'Shapiro p-value', 'Value': float(sh_p)})
    except Exception:
        pass
    try:
        jb_stat, jb_p = jarque_bera(resid)
        regression_diagnostic_rows.append({'Diagnostic': 'Jarque-Bera p-value', 'Value': float(jb_p)})
    except Exception:
        pass
regression_diagnostics_df = pd.DataFrame(regression_diagnostic_rows)
regression_diagnostics_df


In [ ]:
classification_df = features_df[['Year', 'RainfallAnomaly'] + [c for c in feature_candidates if c in features_df.columns]].copy()

classification_df = classification_df.loc[:, ~classification_df.columns.duplicated()].copy()
classification_df = classification_df.dropna(subset=['RainfallAnomaly']).reset_index(drop=True)

rainfall_anomaly_series = classification_df['RainfallAnomaly']
if isinstance(rainfall_anomaly_series, pd.DataFrame):
    rainfall_anomaly_series = rainfall_anomaly_series.iloc[:, 0]

rainfall_anomaly_series = pd.to_numeric(rainfall_anomaly_series, errors='coerce')
classification_df['RainfallAnomaly'] = rainfall_anomaly_series
classification_df['RainfallClass'] = (classification_df['RainfallAnomaly'] >= 0).astype(int)

classification_leakage_features = {
    'Rainfall',
    'RainfallAnomaly',
    'RainfallClass',
    'Rainfalldiff1',
    'Rainfallrolling3',
    'Rainfallrolling5',
    'RainfallAnomalydiff1',
    'RainfallAnomalyrolling3',
    'RainfallAnomalyrolling5',
}

cls_features = [
    c for c in classification_df.columns
    if c not in ['Year']
    and c not in classification_leakage_features
    and pd.api.types.is_numeric_dtype(classification_df[c])
]

classification_results_df = pd.DataFrame()
classification_predictions_df = pd.DataFrame()
conf_df = pd.DataFrame()

if len(classification_df) >= 20 and len(cls_features) > 0:
    Xc = pd.DataFrame(
        SimpleImputer(strategy='median').fit_transform(classification_df[cls_features]),
        columns=cls_features
    )
    yc = classification_df['RainfallClass']

    cls_model = Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(max_iter=2000))
    ])

    cls_tscv = TimeSeriesSplit(n_splits=5)
    rows, pred_rows = [], []

    for fold, (train_idx, test_idx) in enumerate(cls_tscv.split(Xc), start=1):
        X_train, X_test = Xc.iloc[train_idx], Xc.iloc[test_idx]
        y_train, y_test = yc.iloc[train_idx], yc.iloc[test_idx]

        cls_model.fit(X_train, y_train)
        yp = cls_model.predict(X_test)

        rows.append({
            'Fold': fold,
            'Accuracy': accuracy_score(y_test, yp),
            'BalancedAccuracy': balanced_accuracy_score(y_test, yp),
            'F1': f1_score(y_test, yp)
        })

        pred_rows.append(pd.DataFrame({
            'Year': classification_df.iloc[test_idx]['Year'].values,
            'Actual': y_test.values,
            'Predicted': yp,
            'Fold': fold
        }))

    classification_results_df = pd.DataFrame(rows)
    classification_predictions_df = pd.concat(pred_rows, ignore_index=True)

    cm = confusion_matrix(
        classification_predictions_df['Actual'],
        classification_predictions_df['Predicted']
    )
    conf_df = pd.DataFrame(
        cm,
        index=['Actual Dry', 'Actual Wet'],
        columns=['Pred Dry', 'Pred Wet']
    )

    fig, axes = plt.subplots(1, 2, figsize=(15, 6))

    sns.lineplot(
        data=classification_results_df.melt(id_vars='Fold', var_name='Metric', value_name='Score'),
        x='Fold', y='Score', hue='Metric', marker='o', ax=axes[0]
    )
    axes[0].set_title('Rainfall classification performance by fold')
    axes[0].set_ylim(0, 1)

    sns.heatmap(conf_df, annot=True, fmt='d', cmap='Blues', ax=axes[1])
    axes[1].set_title('Confusion matrix')

    save_fig('09_rainfall_classification.png')

classification_results_df


In [ ]:
forecast_df = master_df[['Year', 'IndiaTempAnomaly']].dropna().copy()

forecast_output_df = pd.DataFrame()
forecast_diagnostics_df = pd.DataFrame()

if HAS_STATSMODELS and len(forecast_df) >= 20:
    ts = forecast_df.set_index('Year')['IndiaTempAnomaly'].astype(float)

    try:
        adf_stat, adf_p, *_ = adfuller(ts.dropna())
    except Exception:
        adf_stat, adf_p = np.nan, np.nan

    try:
        arima_model = ARIMA(ts, order=(1, 1, 1)).fit()
        future_years = np.arange(ts.index.max() + 1, ts.index.max() + FORECAST_HORIZON + 1)
        forecast_vals = arima_model.forecast(steps=FORECAST_HORIZON)
        forecast_output_df = pd.DataFrame({
            'Year': future_years,
            'ForecastIndiaTempAnomaly': np.array(forecast_vals)
        })

        try:
            lb = acorr_ljungbox(arima_model.resid.dropna(), lags=[min(10, max(1, len(arima_model.resid.dropna()) // 3))], return_df=True)
            ljung_box_p = float(lb['lb_pvalue'].iloc[0])
        except Exception:
            ljung_box_p = np.nan

        forecast_diagnostics_df = pd.DataFrame([
            {'Diagnostic': 'ADF p-value', 'Value': float(adf_p) if pd.notna(adf_p) else np.nan},
            {'Diagnostic': 'AIC', 'Value': float(arima_model.aic)},
            {'Diagnostic': 'BIC', 'Value': float(arima_model.bic)},
            {'Diagnostic': 'Ljung-Box p-value', 'Value': ljung_box_p},
        ])

        fig, ax = plt.subplots(figsize=(13, 6))
        ax.plot(ts.index, ts.values, marker='o', lw=2.3, label='Observed')
        ax.plot(forecast_output_df['Year'], forecast_output_df['ForecastIndiaTempAnomaly'], marker='s', lw=2.3, label='Forecast')
        ax.set_title('ARIMA forecast of India temperature anomaly')
        ax.set_xlabel('Year')
        ax.set_ylabel('India temperature anomaly')
        ax.legend(frameon=True)
        save_fig('10_arima_forecast.png')

    except Exception as e:
        forecast_diagnostics_df = pd.DataFrame([{'Diagnostic': 'Forecast error', 'Value': str(e)}])

display(forecast_output_df.head())
display(forecast_diagnostics_df)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

if 'ONIMonsoonMean' in master_df.columns and 'RainfallAnomaly' in master_df.columns:
    temp = master_df[['ONIMonsoonMean', 'RainfallAnomaly']].dropna()
    sns.regplot(data=temp, x='ONIMonsoonMean', y='RainfallAnomaly', ax=axes[0], scatter_kws={'s': 70})
    axes[0].set_title('Monsoon ONI vs rainfall anomaly')

if 'ONIAnnualMean' in master_df.columns and 'IndiaTempAnomaly' in master_df.columns:
    temp = master_df[['ONIAnnualMean', 'IndiaTempAnomaly']].dropna()
    sns.regplot(data=temp, x='ONIAnnualMean', y='IndiaTempAnomaly', ax=axes[1], scatter_kws={'s': 70}, color='tab:red')
    axes[1].set_title('Annual ONI vs India temperature anomaly')

save_fig('11_enso_relationships.png')


In [ ]:
if resolved['imd_tmax_annual'] is not None and resolved['imd_tmin_annual'] is not None:
    tmax = clip_years(read_csv_flex(resolved['imd_tmax_annual']))
    tmin = clip_years(read_csv_flex(resolved['imd_tmin_annual']))

    tmax_col = [c for c in tmax.columns if c != 'Year'][0]
    tmin_col = [c for c in tmin.columns if c != 'Year'][0]

    tmax = ensure_numeric(tmax, [tmax_col]).rename(columns={tmax_col: 'TmaxAnnual'})
    tmin = ensure_numeric(tmin, [tmin_col]).rename(columns={tmin_col: 'TminAnnual'})

    diurnal_df = tmax[['Year', 'TmaxAnnual']].merge(
        tmin[['Year', 'TminAnnual']], on='Year', how='inner'
    )
    diurnal_df['DTR'] = diurnal_df['TmaxAnnual'] - diurnal_df['TminAnnual']

    temp_median = np.nanmedian(pd.concat([diurnal_df['TmaxAnnual'], diurnal_df['TminAnnual']]).values)
    imd_unit_label = '°F' if temp_median > 45 else '°C'

    fig, ax1 = plt.subplots(figsize=(13, 6))

    line1 = ax1.plot(diurnal_df['Year'], diurnal_df['TmaxAnnual'], lw=2.3, label=f'Tmax ({imd_unit_label})')
    line2 = ax1.plot(diurnal_df['Year'], diurnal_df['TminAnnual'], lw=2.3, label=f'Tmin ({imd_unit_label})')
    ax1.set_title('IMD Tmax, Tmin and diurnal temperature range')
    ax1.set_xlabel('Year')
    ax1.set_ylabel(f'Temperature ({imd_unit_label})')

    ax2 = ax1.twinx()
    line3 = ax2.plot(diurnal_df['Year'], diurnal_df['DTR'], lw=2.3, color='tab:green', label=f'DTR ({imd_unit_label})')
    ax2.set_ylabel(f'DTR ({imd_unit_label})')

    lines = line1 + line2 + line3
    labels = [l.get_label() for l in lines]
    ax1.legend(lines, labels, frameon=True, loc='best')

    save_fig('13_imd_dtr.png')
else:
    diurnal_df = pd.DataFrame()

diurnal_df.head()


In [ ]:
final_summary_lines = []

if not trend_df.empty:
    for _, row in trend_df.iterrows():
        final_summary_lines.append(
            f"{row['Variable']}: trend={row['Trend']}, slope/year={row['LinearSlopePerYear']:.4f}, "
            f"MK p={row['pvalue']:.4g}" if pd.notna(row['pvalue']) else
            f"{row['Variable']}: trend={row['Trend']}, slope/year={row['LinearSlopePerYear']:.4f}"
        )

if not model_summary_df.empty:
    best_row = model_summary_df.iloc[0]
    final_summary_lines.append(
        f"Best regression model: {best_row['Model']} with mean RMSE={best_row['RMSE']:.4f}, "
        f"MAE={best_row['MAE']:.4f}, R2={best_row['R2']:.4f}"
    )

if not classification_results_df.empty:
    final_summary_lines.append(
        f"Rainfall classification mean accuracy={classification_results_df['Accuracy'].mean():.4f}, "
        f"balanced accuracy={classification_results_df['BalancedAccuracy'].mean():.4f}, "
        f"F1={classification_results_df['F1'].mean():.4f}"
    )

if not forecast_output_df.empty:
    final_summary_lines.append(
        f"Forecast horizon generated: {len(forecast_output_df)} years beyond {forecast_df['Year'].max()}"
    )

final_summary_text = '\n'.join(final_summary_lines)
print(final_summary_text)


In [ ]:
excel_file = OUTPUT_DIR / 'climate_analysis_complete_outputs.xlsx'

if HAS_OPENPYXL:
    with pd.ExcelWriter(excel_file, engine='openpyxl') as writer:
        file_status_df.to_excel(writer, sheet_name=safe_sheet_name('file_status'), index=False)
        master_df.to_excel(writer, sheet_name=safe_sheet_name('master_data'), index=False)
        features_df.to_excel(writer, sheet_name=safe_sheet_name('features'), index=False)
        oni_annual_df.to_excel(writer, sheet_name=safe_sheet_name('oni_annual'), index=False)
        summary_metrics.to_excel(writer, sheet_name=safe_sheet_name('summary_metrics'), index=False)
        trend_df.to_excel(writer, sheet_name=safe_sheet_name('trends'), index=False)
        corr_matrix.to_excel(writer, sheet_name=safe_sheet_name('corr_matrix'))
        explained_variance_df.to_excel(writer, sheet_name=safe_sheet_name('pca_variance'), index=False)
        loadings_df.to_excel(writer, sheet_name=safe_sheet_name('pca_loadings'), index=False)
        pca_scores_df.to_excel(writer, sheet_name=safe_sheet_name('pca_scores'), index=False)
        model_results_df.to_excel(writer, sheet_name=safe_sheet_name('regression_cv'), index=False)
        model_summary_df.to_excel(writer, sheet_name=safe_sheet_name('regression_summary'), index=False)
        predictions_df.to_excel(writer, sheet_name=safe_sheet_name('regression_predictions'), index=False)
        importance_df.to_excel(writer, sheet_name=safe_sheet_name('feature_importance'), index=False)
        regression_diagnostics_df.to_excel(writer, sheet_name=safe_sheet_name('reg_diagnostics'), index=False)
        classification_results_df.to_excel(writer, sheet_name=safe_sheet_name('classification_cv'), index=False)
        classification_predictions_df.to_excel(writer, sheet_name=safe_sheet_name('classification_preds'), index=False)
        conf_df.to_excel(writer, sheet_name=safe_sheet_name('classification_cm'))
        forecast_output_df.to_excel(writer, sheet_name=safe_sheet_name('forecast'), index=False)
        forecast_diagnostics_df.to_excel(writer, sheet_name=safe_sheet_name('forecast_diag'), index=False)
        diurnal_df.to_excel(writer, sheet_name=safe_sheet_name('imd_dtr'), index=False)
        
    print(f'Saved Excel workbook to {excel_file}')
else:
    print('openpyxl not available; Excel export skipped.')

print(f'Saved figures to {FIGURE_DIR}')
print(f'Saved tables to {TABLE_DIR}')
